## Notebook 2 — Preprocessing

## 1. Imports and Load Data

We import the same core libraries as the exploration notebook, plus two new ones:
- `train_test_split` — splits data into training and testing sets
- `StandardScaler` — rescales features to have mean 0 and standard deviation 1

In [5]:
import pandas as pd # data loading and manipulation
import numpy as np # numerical operations
import os # builds file paths for Mac and Windows
from sklearn.model_selection import train_test_split # splits data into train/test
from sklearn.preprocessing import StandardScaler # scales features to mean=0, std=1


# Load the raw dataset
data_path = os.path.join('..', 'data', 'creditcard.csv')
df = pd.read_csv(data_path)

# Recreate Log_Amount — log transform fixes the skewness found in exploration
df['Log_Amount'] = np.log1p(df['Amount'])

# Recreate Hour — converts raw seconds into a 0-24 hour clock
df['Hour'] = (df['Time'] / 3600).mod(24)

## 2. Train/Test Split

Our target is `Class` — 0 for legitimate, 1 for fraud

- `Amount` — replaced by `Log_Amount` which fixes the skewness
- `Time` — replaced by `Hour` which is more meaningful

We split the data into two sets:
- **Training set (80%)** — used to teach the model patterns in the data
- **Test set (20%)** — used to evaluate how well the model performs on unseen data

**Why `stratify=y`?**
only 0.17% of transactions are fraud
Without stratify, a random split might put almost no fraud cases into the test set by chance
With stratify guarantees that both the training and test sets contain the same 0.17% fraud ratio

**Why `random_state=42`?**
Every single run produces the exact same split

In [6]:
# y is the target variable — what we want the model to predict
y = df['Class']  # 0 = legitimate, 1 = fraud

# X is all features — we drop Class (target), Amount (replaced), and Time (replaced)
# Using drop() removes those columns and keeps everything else
X = df.drop(columns=['Class', 'Amount', 'Time'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, # 20% of data goes to test set, 80% to training
    random_state=42,
    stratify=y
)

# Confirm the split sizes
print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

print(f'Fraud ratio in training : {y_train.mean()*100:.4f}%')
print(f'Fraud ratio in test     : {y_test.mean()*100:.4f}%')

Training set size: (227845, 30)
Testing set size: (56962, 30)
Fraud ratio in training : 0.1729%
Fraud ratio in test     : 0.1720%


## 3. Scale Features — StandardScaler

**What does StandardScaler do?**
It rescales each column so that:
- The mean becomes 0
- The standard deviation becomes 1

The formula is: `scaled = (value - mean) / std`

**Why scale at all?**
Logistic Regression treats larger numbers as more important by default,
so `Hour` would unfairly dominate the model just because of its scale.
Scaling puts all features on the same playing field.



In [7]:
scaler = StandardScaler()

# fit() learns the mean and standard deviation (training data only)
scaler.fit(X_train[['Log_Amount', 'Hour']])

# Print what the scaler learned from training data
print("Log_Amount — Mean:", scaler.mean_[0], " Std:", scaler.scale_[0])
print("Hour       — Mean:", scaler.mean_[1], " Std:", scaler.scale_[1])

# transform() applies the scaling using the values learned above
X_train[['Log_Amount', 'Hour']] = scaler.transform(X_train[['Log_Amount', 'Hour']])
# for test data we use transform() only — same scale, does not relearn
X_test[['Log_Amount', 'Hour']]  = scaler.transform(X_test[['Log_Amount', 'Hour']])


Log_Amount — Mean: 3.153058483845931  Std: 1.655991754686043
Hour       — Mean: 14.541464317847659  Std: 5.841764338103793


## 4. Save Processed Data

save all four splits to `data/processed/` as CSV files
means the model notebook can load these files directly without having to redo any preprocessing steps

**Why save as separate files?**
Keeping train and test separate, makes it
impossible to accidentally mix them up in the model notebook

**Why `index=False`?**
We don't need the row numbers that pandas saves 

In [8]:
# Create the processed data folder
# exist_ok=True prevents an error
os.makedirs(os.path.join('..', 'data', 'processed'), exist_ok=True)

# Save all four splits as CSV files
X_train.to_csv(os.path.join('..', 'data', 'processed', 'X_train.csv'), index=False)
X_test.to_csv(os.path.join('..', 'data', 'processed', 'X_test.csv'), index=False)
y_train.to_csv(os.path.join('..', 'data', 'processed', 'y_train.csv'), index=False)
y_test.to_csv(os.path.join('..', 'data', 'processed', 'y_test.csv'), index=False)

# Confirm everything saved correctly
print("Processed data saved to data/processed/")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

Processed data saved to data/processed/
X_train: (227845, 30), X_test: (56962, 30)
y_train: (227845,), y_test: (56962,)
